# 02 · Silver · `ocorrencias_rede`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.ocorrencias_rede` (`ocorrencias_rede.csv`, tudo como texto) e grava `v_energy.silver.ocorrencias_rede`. **Grão: 1 linha = 1 ocorrência de interrupção em 1 UC** (o glossário do case diz que cada registro é uma UC interrompida).

**Fonte das regras:** `exploracao_ocorrencias_rede.ipynb` e a seção `ocorrencias_rede` do `docs/data_quality_log.md` (O1 a O3).

| # | Tratamento | Linhas afetadas | Regra | Log |
|---|---|---|---|---|
| 1 | `causa` ausente em 4 formas (nulo, `-`, `N/A`, `NULL`) | 5.400 | nulo real | O1 |
| 2 | `duracao_min` com o texto ` min` | 16.200 | remover o sufixo e converter para inteiro | O2 |
| 3 | Datas como texto | 180.000 | `timestamp` (ISO com horário) | O3 |

**Nenhuma linha é removida:** a tabela não tem cópias, registros em dobro, órfãs nem UCs de teste (exploração, seções 3 a 8).

**Por que o cuidado:** esta tabela é a base do **DEC/FEC**. A checagem mais importante é `fim − início = duração` em todas as linhas: ela prova que datas e duração estão coerentes.

**Resultado esperado:** 180.000 linhas, `id_ocorrencia` único.

**Pré-requisito:** `silver.unidades_consumidoras` gravada (integridade e alimentador).

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.ocorrencias_rede"
tabela_silver = f"{silver_schema}.ocorrencias_rede"

def n_br(n):
    return f"{n:,}".replace(",", ".")

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

## 2. Funções de checagem de qualidade

Mesmas funções das outras Silvers, gravando em `silver.dq_log`. O log é gravado **sempre**; a Silver só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Leitura sem transformação. Conferência: **180.000** linhas, todas as colunas `string`.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 180.000

bronze.printSchema()
display(bronze.limit(5))

## 4. `causa` ausente em quatro formas (log O1)

**Problema:** a causa aparece vazia de 4 jeitos: nulo real (1.356), `-` (1.363), `N/A` (1.341) e `NULL` (1.340). Total: **5.400 linhas (3,0%)**.

**Decisão:** as 4 formas viram **nulo real**, como os códigos de ausência de leituras (L3). A ocorrência continua válida (tem UC, datas e duração) e entra no DEC/FEC.

**Atenção:** `NAO IDENTIFICADA` (26.328) **não** é ausência: é uma causa registrada, investigada sem conclusão. Ela é mantida como está.

**Conferência:** 5.400 causas nulas; 6 causas válidas.

In [0]:
# Tratamento 1 · causa: 4 formas de ausência viram nulo real
causa_limpa = F.upper(F.trim(F.col("causa")))
df_causa = bronze.withColumn(
    "causa", F.when(causa_limpa.isin("-", "N/A", "NULL", ""), F.lit(None).cast("string")).otherwise(causa_limpa))

# esperado: 5.400 nulas | 6 causas válidas
print("causa nula:", n_br(df_causa.filter("causa IS NULL").count()))
display(df_causa.groupBy("causa").count().orderBy(F.desc("count")))

## 5. `duracao_min` com o texto " min" (log O2)

**Problema:** 16.200 durações (9,0%) vêm com o texto ` min` junto do número (`28 min`, `127 min`). Um `cast` direto anularia todas.

**Decisão:** remover o sufixo e converter para **inteiro** (todas as durações são minutos inteiros). O valor original fica em `duracao_min_original` só no notebook.

**Validação:** na exploração, `fim − início` bateu com a duração em 100% das 163.800 linhas numéricas. A checagem da seção 9 repete o teste nas **180.000** linhas, incluindo as 16.200 corrigidas.

**Conferência:** 16.200 com sufixo; 0 nulas depois da conversão.

In [0]:
# Tratamento 2 · duração: remove o sufixo " min" e converte para inteiro
df_dur = (
    df_causa
    .withColumn("duracao_min_original", F.col("duracao_min"))
    .withColumn("duracao_min", F.expr("try_cast(trim(regexp_replace(duracao_min, '(?i)min', '')) as int)"))
)

# esperado: 16.200 | 0
print("com sufixo ' min':        ", n_br(df_dur.filter(F.lower(F.col("duracao_min_original")).contains("min")).count()))
print("duração nula após conversão:", n_br(df_dur.filter("duracao_min IS NULL").count()))
display(df_dur.filter(F.lower(F.col("duracao_min_original")).contains("min"))
        .select("id_ocorrencia", "duracao_min_original", "duracao_min").limit(5))

## 6. Datas e textos (log O3)

- `data_abertura` e `data_encerramento`: sempre ISO com horário (`yyyy-MM-dd HH:mm:ss`), sem sentinela e sem fim ausente. Viram `timestamp`, **mantendo o horário** (necessário para a duração e para a visão 360).
- `tipo`, `causa` e `alimentador`: já sem variações; `upper(trim)` só garante o padrão.
- `mes_referencia` (`yyyy-MM` da abertura): derivada aqui porque é a chave de mês usada em todas as Silvers e no DEC/FEC mensal.

**Conferência:** 0 datas nulas.

In [0]:
# Tratamento 3 · datas com horário e textos padronizados
df_dt = (
    df_dur
    .withColumn("data_abertura", F.expr("try_to_timestamp(trim(data_abertura), 'yyyy-MM-dd HH:mm:ss')"))
    .withColumn("data_encerramento", F.expr("try_to_timestamp(trim(data_encerramento), 'yyyy-MM-dd HH:mm:ss')"))
    .withColumn("mes_referencia", F.date_format("data_abertura", "yyyy-MM"))
    .withColumn("tipo", F.upper(F.trim("tipo")))
    .withColumn("alimentador", F.upper(F.trim("alimentador")))
    .withColumn("id_uc", F.upper(F.trim("id_uc")))
    .withColumn("id_ocorrencia", F.upper(F.trim("id_ocorrencia")))
)

# esperado: 0 | 0
print("data_abertura nula:    ", n_br(df_dt.filter("data_abertura IS NULL").count()))
print("data_encerramento nula:", n_br(df_dt.filter("data_encerramento IS NULL").count()))
display(df_dt.groupBy(F.year("data_abertura").alias("ano")).count().orderBy("ano"))

## 7. Seleção final das colunas

`duracao_min_original` fica de fora. `source_file` é mantida para auditoria.

In [0]:
colunas_finais = [
    "id_ocorrencia", "id_uc", "alimentador", "tipo", "causa",
    "data_abertura", "data_encerramento", "mes_referencia", "duracao_min", "source_file",
]
df_silver = df_dt.select(*colunas_finais)
df_silver.printSchema()

## 8. Checagens de qualidade

| Checagem | Protege |
|---|---|
| Total = 180.000 e `id_ocorrencia` único | nenhuma linha perdida ou duplicada |
| Mesma UC + mesma abertura só uma vez | DEC/FEC não conta a mesma interrupção duas vezes |
| Colunas obrigatórias não nulas | conversões (O2, O3) |
| `id_uc` no padrão e existente na Silver de UCs | integridade com o cadastro |
| Alimentador igual ao do cadastro da UC | o DEC/FEC por alimentador não depende de qual fonte usar |
| `tipo` entre os 5 valores; `causa` entre os 6 ou nula | domínios (O1) |
| **`fim − início = duração`** (tolerância de 1 min) | coerência de datas e duração (O2, O3) |
| Duração entre 1 minuto e 7 dias; fim não antes do início | valores impossíveis |
| Abertura entre 2022-01-01 e 2026-06-30 | período da extração |

A contagem de 180.000 vale para esta extração; com dados novos, o valor esperado muda.

In [0]:
t = "silver.ocorrencias_rede"
dq_results.clear()

def ok(cond):
    return F.coalesce(cond, F.lit(False))

dq_check_contagem(t, "total de linhas = 180.000", df_silver, 180_000)
dq_check_unique(t, "id_ocorrencia único", df_silver, ["id_ocorrencia"])
dq_check_unique(t, "mesma UC e mesma abertura só uma vez", df_silver, ["id_uc", "data_abertura"])

for coluna in ["id_ocorrencia", "id_uc", "alimentador", "tipo", "data_abertura", "data_encerramento",
               "mes_referencia", "duracao_min"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, ok(F.col("id_uc").rlike("^UC[0-9]{6}$")))
dq_check(t, "alimentador no padrão ALM-XXX-000", df_silver, ok(F.col("alimentador").rlike("^ALM-[A-Z]{3}-[0-9]{3}$")))

ucs = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc", F.col("alimentador").alias("alim_cadastro"))
com_uc = df_silver.join(ucs, "id_uc", "left")
dq_check(t, "id_uc existe na Silver de UCs", com_uc, F.col("alim_cadastro").isNotNull())
dq_check(t, "alimentador igual ao do cadastro da UC", com_uc, ok(F.col("alimentador") == F.col("alim_cadastro")))

dq_check(t, "tipo entre os 5 valores", df_silver,
         ok(F.col("tipo").isin("FALTA DE ENERGIA", "DANO EM EQUIPAMENTO", "OSCILACAO", "SOBRECARGA", "FIO PARTIDO")))
dq_check(t, "causa entre os 6 valores ou nula", df_silver,
         F.col("causa").isNull() | ok(F.col("causa").isin("CLIMATICA", "SOBRECARGA", "NAO IDENTIFICADA",
                                                           "VEGETACAO", "TERCEIROS", "FALHA DE EQUIPAMENTO")))

dif_min = (F.unix_timestamp("data_encerramento") - F.unix_timestamp("data_abertura")) / 60
dq_check(t, "fim - início = duração (tolerância de 1 min)", df_silver, ok(F.abs(dif_min - F.col("duracao_min")) <= 1))
dq_check(t, "duração entre 1 minuto e 7 dias", df_silver, ok(F.col("duracao_min").between(1, 10080)))
dq_check(t, "fim não antes do início", df_silver, ok(F.col("data_encerramento") >= F.col("data_abertura")))
dq_check(t, "abertura entre 2022-01-01 e 2026-06-30", df_silver,
         ok(F.col("data_abertura").between(F.lit("2022-01-01 00:00:00").cast("timestamp"),
                                           F.lit("2026-06-30 23:59:59").cast("timestamp"))))

## 9. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`).

In [0]:
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == t)
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

## 10. Gravação da Silver, só se todas as checagens passarem

In [0]:
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

## 11. Bronze × Silver, com as mesmas ocorrências

Para cada tratamento, os mesmos `id_ocorrencia` nas duas tabelas: o texto original da Bronze ao lado do valor tratado.

In [0]:
silver_t = spark.table(tabela_silver)
cols_ver = ["id_ocorrencia", "id_uc", "tipo", "causa", "data_abertura", "data_encerramento", "duracao_min"]

def comparar(titulo, ids_df, n=4):
    ids = [r["id_ocorrencia"] for r in ids_df.select("id_ocorrencia").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_ocorrencia").isin(ids)).select(*cols_ver).orderBy("id_ocorrencia"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_ocorrencia").isin(ids)).select(*cols_ver, "mes_referencia").orderBy("id_ocorrencia"))

# uma de cada forma de ausência da causa
comparar("Causa ausente (uma de cada forma)",
         bronze.withColumn("forma", F.coalesce(F.trim("causa"), F.lit("<nulo>")))
               .filter(F.col("forma").isin("<nulo>", "-", "N/A", "NULL")).dropDuplicates(["forma"]))
comparar("Duração com ' min'", bronze.filter(F.lower(F.col("duracao_min")).contains("min")), n=3)